# Complete Inference & Evaluation Pipeline
## Preprocess Data -> Load Models -> Evaluate

This notebook downloads datasets, preprocesses them, and evaluates both trained models.

## 1. Setup & Dependencies

In [60]:
import pandas as pd
import torch
from torch.utils.data import DataLoader
import numpy as np
from transformers import AutoTokenizer
import os
from sklearn.model_selection import train_test_split
from huggingface_hub import hf_hub_download

# Hardware optimization
torch.backends.cuda.matmul.allow_tf32 = True
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"CUDA Available: {torch.cuda.is_available()}")

Device: cuda
CUDA Available: True


## 2. Import Custom Modules

In [61]:
# Import custom modules
import dataset
import model_utils
import eval as ev
import validate as val

# Reload modules
import importlib
importlib.reload(dataset)
importlib.reload(model_utils)
importlib.reload(ev)
importlib.reload(val)

from dataset import TicketDataset
from model_utils import MultiTaskXLMR

print("Custom modules imported successfully!")

Custom modules imported successfully!


## 3. Define Model Repos from Hugging Face

In [62]:
# Hugging Face repository config
HUGGINGFACE_USERNAME = "pshashid"  # ← CHANGE THIS to your HF username
REPO_NAME_ENGLISH = "xlmr-toxicity-english"
REPO_NAME_MULTILINGUAL = "xlmr-toxicity-multilingual"

ENGLISH_REPO_ID = f"{HUGGINGFACE_USERNAME}/{REPO_NAME_ENGLISH}"
MULTILINGUAL_REPO_ID = f"{HUGGINGFACE_USERNAME}/{REPO_NAME_MULTILINGUAL}"

# Model filename inside the HF repos
ENGLISH_MODEL_FILENAME = "xlmr_toxicity_english.bin"
MULTILINGUAL_MODEL_FILENAME = "xlmr_toxicity_multilingual.bin"

print(f"English repo   : {ENGLISH_REPO_ID}")
print(f"Multilingual repo: {MULTILINGUAL_REPO_ID}")

# Create data directories
os.makedirs("data/processed/jigsaw", exist_ok=True)
os.makedirs("data/processed/multilingual_toxic", exist_ok=True)
print("\nData directories created!")

English repo   : pshashid/xlmr-toxicity-english
Multilingual repo: pshashid/xlmr-toxicity-multilingual

Data directories created!


## 4. Preprocess Jigsaw English Dataset

In [63]:
print("PREPROCESSING JIGSAW ENGLISH DATASET")

from datasets import load_dataset

print("\nDownloading Jigsaw English dataset...")
jigsaw = load_dataset("thesofakillers/jigsaw-toxic-comment-classification-challenge")

train_df = jigsaw["train"].to_pandas()
test_df = jigsaw["test"].to_pandas()

label_cols = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

print(f"\nOriginal dataset sizes:")
print(f"  Train: {len(train_df)}")
print(f"  Test: {len(test_df)}")

# Check label distribution
print(f"\nLabel distribution (train set):")
label_counts = train_df[label_cols].sum()
for col, count in label_counts.items():
    pct = (count / len(train_df)) * 100
    print(f"  {col:<15}: {count:>6} ({pct:>5.2f}%)")

# Clean text
print("\nCleaning text...")
for df in [train_df, test_df]:
    df["comment_text"] = df["comment_text"].astype(str).str.strip().str.replace("\n", " ")

# Train/validation/test split
print("\nSplitting dataset (80% train, 10% val, 10% test)...")
train_df, temp_df = train_test_split(
    train_df,
    test_size=0.2,
    random_state=42,
    stratify=train_df[label_cols].apply(lambda x: x.any(), axis=1)
)

val_df, test_split_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42,
    stratify=temp_df[label_cols].apply(lambda x: x.any(), axis=1)
)

# Save CSVs
train_csv = "data/processed/jigsaw/jigsaw_train.csv"
val_csv = "data/processed/jigsaw/jigsaw_val.csv"
test_csv = "data/processed/jigsaw/jigsaw_test.csv"

train_df.to_csv(train_csv, index=False)
val_df.to_csv(val_csv, index=False)
test_split_df.to_csv(test_csv, index=False)

print(f"\n Saved preprocessed data:")
print(f"  Train: {len(train_df)} samples - {train_csv}")
print(f"  Val: {len(val_df)} samples - {val_csv}")
print(f"  Test: {len(test_split_df)} samples - {test_csv}")

PREPROCESSING JIGSAW ENGLISH DATASET


Original dataset sizes:
  Train: 159571
  Test: 306328

Label distribution (train set):
  toxic          :  15294 ( 9.58%)
  severe_toxic   :   1595 ( 1.00%)
  obscene        :   8449 ( 5.29%)
  threat         :    478 ( 0.30%)
  insult         :   7877 ( 4.94%)
  identity_hate  :   1405 ( 0.88%)

Cleaning text...

Splitting dataset (80% train, 10% val, 10% test)...

 Saved preprocessed data:
  Train: 127656 samples - data/processed/jigsaw/jigsaw_train.csv
  Val: 15957 samples - data/processed/jigsaw/jigsaw_val.csv
  Test: 15958 samples - data/processed/jigsaw/jigsaw_test.csv


## 5. Preprocess Multilingual Toxic Dataset

In [64]:
print("PREPROCESSING MULTILINGUAL TOXICITY DATASET")

print("\nDownloading TextDetox multilingual dataset...")
multilingual_dataset = load_dataset("textdetox/multilingual_toxicity_dataset")

print(f"\nAvailable languages: {list(multilingual_dataset.keys())}")

all_non_en = []
lang_counts = {}

# Process non-English languages
for lang, ds in multilingual_dataset.items():
    if lang == "en":
        continue

    df = ds.to_pandas()
    df["text"] = df["text"].astype(str).str.strip().str.replace("\n", " ")
    df["language"] = lang
    all_non_en.append(df)
    lang_counts[lang] = len(df)
    print(f"  {lang:<5}: {len(df):>6} samples, Toxic: {df['toxic'].sum():>6}")

# Merge all non-English
print("\nMerging all non-English languages...")
merged_df = pd.concat(all_non_en, ignore_index=True)
print(f"Total merged samples: {len(merged_df)}")
print(f"Toxic samples: {merged_df['toxic'].sum()} ({(merged_df['toxic'].sum()/len(merged_df)*100):.2f}%)")

# 80/20 train/test split
print("\nSplitting into train (80%) and test (20%)...")
if merged_df["toxic"].nunique() > 1:
    train_df_multi, test_df_multi = train_test_split(
        merged_df,
        test_size=0.2,
        stratify=merged_df["toxic"],
        random_state=42
    )
else:
    train_df_multi = merged_df
    test_df_multi = pd.DataFrame(columns=merged_df.columns)

# Save merged CSVs
train_path = "data/processed/multilingual_toxic/merged_non_en_train.csv"
test_path = "data/processed/multilingual_toxic/merged_non_en_test.csv"

train_df_multi.to_csv(train_path, index=False)
test_df_multi.to_csv(test_path, index=False)

print(f"\n Saved multilingual data:")
print(f"  Train: {len(train_df_multi)} samples → {train_path}")
print(f"  Test: {len(test_df_multi)} samples → {test_path}")

PREPROCESSING MULTILINGUAL TOXICITY DATASET


Available languages: ['en', 'ru', 'uk', 'de', 'es', 'am', 'zh', 'ar', 'hi', 'it', 'fr', 'he', 'hin', 'tt', 'ja']
  ru   :   5000 samples, Toxic:   2500
  uk   :   5000 samples, Toxic:   2500
  de   :   5000 samples, Toxic:   2500
  es   :   5000 samples, Toxic:   2500
  am   :   5000 samples, Toxic:   2500
  zh   :   5000 samples, Toxic:   2500
  ar   :   5000 samples, Toxic:   2500
  hi   :   5000 samples, Toxic:   2500
  it   :   5000 samples, Toxic:   2500
  fr   :   5000 samples, Toxic:   2500
  he   :   2011 samples, Toxic:    807
  hin  :   4363 samples, Toxic:   2200
  tt   :   5000 samples, Toxic:   2500
  ja   :   5000 samples, Toxic:   2500

Merging all non-English languages...
Total merged samples: 66374
Toxic samples: 33007 (49.73%)

Splitting into train (80%) and test (20%)...

 Saved multilingual data:
  Train: 53099 samples → data/processed/multilingual_toxic/merged_non_en_train.csv
  Test: 13275 samples → data/processed/mult

## 6. Initialize Model

In [65]:
model_name = "xlm-roberta-large"
num_intents = 5  # severe_toxic, obscene, threat, insult, identity_hate

# Create model instance
model = MultiTaskXLMR(model_name=model_name, num_intents=num_intents).to(device)
print(f" Model created: {model_name}")
print(f" Model moved to {device}")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-large
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


 Model created: xlm-roberta-large
 Model moved to cuda


## 7. Evaluate English Model on Jigsaw Test Set

In [66]:
# Load the English model
print("ENGLISH MODEL EVALUATION")

print("\nDownloading English model from Hugging Face...")
english_model_path = hf_hub_download(
    repo_id=ENGLISH_REPO_ID,
    filename=ENGLISH_MODEL_FILENAME
)
model.load_state_dict(torch.load(english_model_path, map_location=device))
model.to(device)
model.eval()
print("English model loaded successfully!")

# Load English test data
print("\nLoading preprocessed Jigsaw test dataset...")
en_test_df = pd.read_csv("data/processed/jigsaw/jigsaw_test.csv")

# Sample if too large for faster evaluation
if len(en_test_df) > 2000:
    en_test_df = en_test_df.sample(2000, random_state=42)
    print(f"Sampled to 2000 for faster evaluation")

# Prepare labels
intent_cols = ['severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']
en_labels = [
    {'toxic': t, 'intents': i.tolist()}
    for t, i in zip(en_test_df['toxic'].values, en_test_df[intent_cols].values)
]

# Create dataset and loader
en_test_ds = TicketDataset(en_test_df['comment_text'].values, en_labels, model_name=model_name)
en_test_loader = DataLoader(en_test_ds, batch_size=64, shuffle=False)

print(f"\n Test set loaded:")
print(f"  Total samples: {len(en_test_df)}")
print(f"  Toxic: {en_test_df['toxic'].sum()}")
print(f"  Clean: {(1 - en_test_df['toxic']).sum()}")

ENGLISH MODEL EVALUATION

English model loaded successfully!

Loading preprocessed Jigsaw test dataset...
Sampled to 2000 for faster evaluation

 Test set loaded:
  Total samples: 2000
  Toxic: 203
  Clean: 1797


In [67]:
# Evaluate English model on toxicity classification
print("TOXICITY CLASSIFICATION METRICS")

report, auc, lang_f1s, cm = ev.run_evaluation(
    model,
    en_test_loader,
    device,
    lang_list=None
)

print(report)
print(f"\nGLOBAL AUC-ROC: {auc:.4f}")
print("\nCONFUSION MATRIX:")
print(cm)

TOXICITY CLASSIFICATION METRICS


Evaluating: 100%|██████████| 32/32 [00:06<00:00,  5.04it/s]

              precision    recall  f1-score   support

       Clean       0.99      0.98      0.99      1797
       Toxic       0.85      0.91      0.88       203

    accuracy                           0.97      2000
   macro avg       0.92      0.95      0.93      2000
weighted avg       0.98      0.97      0.98      2000


GLOBAL AUC-ROC: 0.9907

CONFUSION MATRIX:
[[1765   32]
 [  18  185]]


In [68]:
# Evaluate English model on intent classification
print("INTENT CLASSIFICATION METRICS")

val.validate_intents(model, en_test_loader, device)

INTENT CLASSIFICATION METRICS


100%|██████████| 32/32 [00:06<00:00,  5.08it/s]


--- DEBUG: First 5 Rows (Post-Fix) ---
Sample 0 Labels: [0. 0. 0. 0. 0.]
Sample 0 Preds : [0.00093994 0.00113357 0.00091105 0.00113357 0.00088305]
Sample 1 Labels: [0. 0. 0. 0. 0.]
Sample 1 Preds : [0.00058841 0.00113357 0.00062633 0.0010005  0.00068785]
Sample 2 Labels: [0. 0. 0. 0. 0.]
Sample 2 Preds : [0.00044422 0.00186757 0.00053578 0.0021157  0.00109873]
Sample 3 Labels: [0. 0. 0. 0. 0.]
Sample 3 Preds : [0.0010005  0.00113357 0.0010005  0.00109873 0.00091105]
Sample 4 Labels: [0. 0. 0. 0. 0.]
Sample 4 Preds : [0.00096975 0.00120659 0.00093994 0.00116951 0.00093994]

--- FINAL PER-INTENT ROC-AUC ---
SEVERE_TOXIC    : 0.9653
OBSCENE         : 0.9864
THREAT          : 0.9678
INSULT          : 0.9819
IDENTITY_HATE   : 0.9780


## 8. Evaluate Multilingual Model on Non-English Test Set

In [69]:
# Load the multilingual model
print("MULTILINGUAL MODEL EVALUATION")

print("\nDownloading Multilingual model from Hugging Face...")
multilingual_model_path = hf_hub_download(
    repo_id=MULTILINGUAL_REPO_ID,
    filename=MULTILINGUAL_MODEL_FILENAME
)
model.load_state_dict(torch.load(multilingual_model_path, map_location=device))
model.to(device)
model.eval()
print("Multilingual model loaded successfully!")

# Load multilingual test data
print("\nLoading preprocessed multilingual test dataset...")
multi_test_df = pd.read_csv("data/processed/multilingual_toxic/merged_non_en_test.csv")

# Sample if too large
if len(multi_test_df) > 2000:
    multi_test_df = multi_test_df.sample(2000, random_state=42)
    print(f"Sampled to 2000 for faster evaluation")

# Prepare labels (intents are [0]*5 since we're only evaluating toxicity)
multi_labels = [{'toxic': t, 'intents': [0]*5} for t in multi_test_df['toxic'].values]

# Create dataset and loader
multi_test_ds = TicketDataset(multi_test_df['text'].values, multi_labels, model_name=model_name)
multi_test_loader = DataLoader(multi_test_ds, batch_size=64, shuffle=False)

print(f"\n Multilingual test set loaded:")
print(f"  Total samples: {len(multi_test_df)}")
print(f"  Toxic: {multi_test_df['toxic'].sum()}")
print(f"  Clean: {(1 - multi_test_df['toxic']).sum()}")
print(f"  Languages: {multi_test_df['language'].nunique()}")
print(f"  {multi_test_df['language'].unique()}")

MULTILINGUAL MODEL EVALUATION

Multilingual model loaded successfully!

Loading preprocessed multilingual test dataset...
Sampled to 2000 for faster evaluation

 Multilingual test set loaded:
  Total samples: 2000
  Toxic: 1007
  Clean: 993
  Languages: 14
  ['am' 'ja' 'it' 'zh' 'de' 'uk' 'ru' 'hi' 'es' 'ar' 'hin' 'fr' 'tt' 'he']


In [70]:
# Evaluate multilingual model with per-language breakdown
print("MULTILINGUAL TOXICITY CLASSIFICATION METRICS")

report, auc, lang_f1s, cm = ev.run_evaluation(
    model,
    multi_test_loader,
    device,
    lang_list=multi_test_df['language'].values
)

print(report)
print(f"\nGLOBAL AUC-ROC: {auc:.4f}")

print("\nPER-LANGUAGE F1 SCORES (Ascending Order):")
print("-" * 40)
for lang in sorted(lang_f1s, key=lang_f1s.get):
    print(f"{lang.upper():<15}: {lang_f1s[lang]:.4f}")

print("\nCONFUSION MATRIX:")
print(cm)

MULTILINGUAL TOXICITY CLASSIFICATION METRICS


Evaluating: 100%|██████████| 32/32 [00:05<00:00,  5.47it/s]

              precision    recall  f1-score   support

       Clean       0.88      0.88      0.88       993
       Toxic       0.88      0.88      0.88      1007

    accuracy                           0.88      2000
   macro avg       0.88      0.88      0.88      2000
weighted avg       0.88      0.88      0.88      2000


GLOBAL AUC-ROC: 0.9485

PER-LANGUAGE F1 SCORES (Ascending Order):
----------------------------------------
HIN            : 0.6952
HE             : 0.7829
ZH             : 0.8150
AM             : 0.8267
AR             : 0.8310
TT             : 0.8741
DE             : 0.8823
ES             : 0.8846
IT             : 0.8904
JA             : 0.8917
UK             : 0.9401
HI             : 0.9403
RU             : 0.9877
FR             : 0.9931

CONFUSION MATRIX:
[[872 121]
 [118 889]]


## 9. Direct Inference Examples

### 9.1 English Model Inference

In [71]:
# Load English model for inference
# english_model_path was already downloaded in cell 7 above
model.load_state_dict(torch.load(english_model_path, map_location=device))
model.to(device)
model.eval()

tokenizer = AutoTokenizer.from_pretrained(model_name)

print("ENGLISH MODEL - INFERENCE EXAMPLES")

intent_names = ['severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

# Example 1: Toxic sentence with threat and insult
test_text_1 = "You are a complete idiot and I will make your life hell!"
inputs_1 = tokenizer(test_text_1, return_tensors="pt", padding=True, truncation=True).to(device)

with torch.no_grad():
    with torch.amp.autocast('cuda', dtype=torch.bfloat16):
        tox_logits_1, intent_logits_1 = model(inputs_1['input_ids'], inputs_1['attention_mask'])

tox_prob_1 = torch.sigmoid(tox_logits_1.float()).item()
intent_probs_1 = torch.sigmoid(intent_logits_1.float()).cpu().numpy().flatten()

print(f"\n[Example 1] '{test_text_1}'")
print(f"Toxicity Probability: {tox_prob_1:.4f}")
print(f"Intent Probabilities:")
for name, prob in zip(intent_names, intent_probs_1):
    print(f"  {name:<15}: {prob:.4f}")

# Example 2: Clean sentence
test_text_2 = "Hello, how are you doing today?"
inputs_2 = tokenizer(test_text_2, return_tensors="pt", padding=True, truncation=True).to(device)

with torch.no_grad():
    with torch.amp.autocast('cuda', dtype=torch.bfloat16):
        tox_logits_2, intent_logits_2 = model(inputs_2['input_ids'], inputs_2['attention_mask'])

tox_prob_2 = torch.sigmoid(tox_logits_2.float()).item()
intent_probs_2 = torch.sigmoid(intent_logits_2.float()).cpu().numpy().flatten()

print(f"\n[Example 2] '{test_text_2}'")
print(f"Toxicity Probability: {tox_prob_2:.4f}")
print(f"Intent Probabilities:")
for name, prob in zip(intent_names, intent_probs_2):
    print(f"  {name:<15}: {prob:.4f}")

ENGLISH MODEL - INFERENCE EXAMPLES

[Example 1] 'You are a complete idiot and I will make your life hell!'
Toxicity Probability: 0.9989
Intent Probabilities:
  severe_toxic   : 0.2465
  obscene        : 0.8661
  threat         : 0.1294
  insult         : 0.9553
  identity_hate  : 0.3585

[Example 2] 'Hello, how are you doing today?'
Toxicity Probability: 0.0003
Intent Probabilities:
  severe_toxic   : 0.0009
  obscene        : 0.0011
  threat         : 0.0008
  insult         : 0.0011
  identity_hate  : 0.0009


### 9.2 Multilingual Model Inference

In [72]:
# Load multilingual model for inference
# multilingual_model_path was already downloaded in cell 8 above
model.load_state_dict(torch.load(multilingual_model_path, map_location=device))
model.to(device)
model.eval()

print("MULTILINGUAL MODEL - INFERENCE EXAMPLES")

# Example 1: German toxic sentence
test_text_de = "Du bist ein Idiot und ich werde dein Leben zur Hölle machen!"  # Toxic in German
inputs_de = tokenizer(test_text_de, return_tensors="pt", padding=True, truncation=True).to(device)

with torch.no_grad():
    with torch.amp.autocast('cuda', dtype=torch.bfloat16):
        tox_logits_de, _ = model(inputs_de['input_ids'], inputs_de['attention_mask'])

tox_prob_de = torch.sigmoid(tox_logits_de.float()).item()

print(f"\n[German - Toxic] '{test_text_de}'")
print(f"Toxicity Probability: {tox_prob_de:.4f}")

# Example 2: Spanish clean sentence
test_text_es = "Hola, ¿cómo estás hoy?"  # Clean in Spanish
inputs_es = tokenizer(test_text_es, return_tensors="pt", padding=True, truncation=True).to(device)

with torch.no_grad():
    with torch.amp.autocast('cuda', dtype=torch.bfloat16):
        tox_logits_es, _ = model(inputs_es['input_ids'], inputs_es['attention_mask'])

tox_prob_es = torch.sigmoid(tox_logits_es.float()).item()

print(f"\n[Spanish - Clean] '{test_text_es}'")
print(f"Toxicity Probability: {tox_prob_es:.4f}")

# Example 3: French toxic sentence
test_text_fr = "Tu es un idiot et je vais te détruire!"  # Toxic in French
inputs_fr = tokenizer(test_text_fr, return_tensors="pt", padding=True, truncation=True).to(device)

with torch.no_grad():
    with torch.amp.autocast('cuda', dtype=torch.bfloat16):
        tox_logits_fr, _ = model(inputs_fr['input_ids'], inputs_fr['attention_mask'])

tox_prob_fr = torch.sigmoid(tox_logits_fr.float()).item()

print(f"\n[French - Toxic] '{test_text_fr}'")
print(f"Toxicity Probability: {tox_prob_fr:.4f}")

# Example 4: Italian clean sentence
test_text_it = "Ciao, come stai oggi?"  # Clean in Italian
inputs_it = tokenizer(test_text_it, return_tensors="pt", padding=True, truncation=True).to(device)

with torch.no_grad():
    with torch.amp.autocast('cuda', dtype=torch.bfloat16):
        tox_logits_it, _ = model(inputs_it['input_ids'], inputs_it['attention_mask'])

tox_prob_it = torch.sigmoid(tox_logits_it.float()).item()

print(f"\n[Italian - Clean] '{test_text_it}'")
print(f"Toxicity Probability: {tox_prob_it:.4f}")

MULTILINGUAL MODEL - INFERENCE EXAMPLES

[German - Toxic] 'Du bist ein Idiot und ich werde dein Leben zur Hölle machen!'
Toxicity Probability: 0.9994

[Spanish - Clean] 'Hola, ¿cómo estás hoy?'
Toxicity Probability: 0.0009

[French - Toxic] 'Tu es un idiot et je vais te détruire!'
Toxicity Probability: 0.9989

[Italian - Clean] 'Ciao, come stai oggi?'
Toxicity Probability: 0.0010


## 10. Batch Inference Helper Function

In [73]:
def batch_inference(texts, model, tokenizer, device, model_type='multilingual'):
    """
    Perform batch inference on multiple texts.

    Args:
        texts: List of text strings
        model: Loaded model
        tokenizer: AutoTokenizer
        device: torch device
        model_type: 'english' or 'multilingual'

    Returns:
        DataFrame with toxicity predictions
    """
    model.eval()
    results = []
    intent_names = ['severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

    with torch.no_grad():
        for text in texts:
            inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True).to(device)

            with torch.amp.autocast('cuda', dtype=torch.bfloat16):
                tox_logits, intent_logits = model(inputs['input_ids'], inputs['attention_mask'])

            tox_prob = torch.sigmoid(tox_logits.float()).item()
            tox_pred = 1 if tox_prob > 0.5 else 0

            result = {
                'text': text,
                'toxicity_prob': tox_prob,
                'is_toxic': tox_pred
            }

            if model_type == 'english':
                intent_probs = torch.sigmoid(intent_logits.float()).cpu().numpy().flatten()
                for name, prob in zip(intent_names, intent_probs):
                    result[name] = prob

            results.append(result)

    return pd.DataFrame(results)

print("Batch inference function defined!")

Batch inference function defined!


In [74]:
# Example: Batch inference with custom texts using multilingual model
test_texts = [
    "You are awesome!",
    "I hate you, you're disgusting!",
    "The weather is nice today.",
    "I'm going to destroy you!",
    "Great job on your presentation!"
]

# Ensure multilingual model is loaded
# multilingual_model_path was already downloaded in cell 8 above
model.load_state_dict(torch.load(multilingual_model_path, map_location=device))
model.to(device)

results_df = batch_inference(test_texts, model, tokenizer, device, model_type='multilingual')

print("BATCH INFERENCE RESULTS (Multilingual Model)")
print(results_df.to_string())

BATCH INFERENCE RESULTS (Multilingual Model)
                              text  toxicity_prob  is_toxic
0                 You are awesome!       0.001598         0
1   I hate you, you're disgusting!       0.993307         1
2       The weather is nice today.       0.000911         0
3        I'm going to destroy you!       0.951863         1
4  Great job on your presentation!       0.000970         0


## 11. Summary & Results

In [75]:
print("PIPELINE SUMMARY")

print("\nCOMPLETED STEPS:")
print("  1. Downloaded and preprocessed Jigsaw English dataset")
print("     - Train: {} samples".format(len(train_df)))
print("     - Val: {} samples".format(len(val_df)))
print("     - Test: {} samples".format(len(test_split_df)))

print("\n  2. Downloaded and preprocessed Multilingual Toxic dataset")
print("     - Train: {} samples".format(len(train_df_multi)))
print("     - Test: {} samples".format(len(test_df_multi)))

print("\n  3. Loaded pre-trained models from Hugging Face")
print(f"     - English model      : {ENGLISH_REPO_ID}/{ENGLISH_MODEL_FILENAME}")
print(f"     - Multilingual model : {MULTILINGUAL_REPO_ID}/{MULTILINGUAL_MODEL_FILENAME}")

print("\n  4. Evaluated both models on test sets")
print("     - English toxicity classification metrics")
print("     - English intent classification metrics")
print("     - Multilingual toxicity classification metrics")
print("     - Per-language F1 scores")

print("\n  5. Performed inference on sample texts")
print("     - English toxic/clean examples")
print("     - Multilingual examples (German, Spanish, French, Italian)")
print("     - Batch inference on custom text list")

print("PIPELINE COMPLETE!")


PIPELINE SUMMARY

COMPLETED STEPS:
  1. Downloaded and preprocessed Jigsaw English dataset
     - Train: 127656 samples
     - Val: 15957 samples
     - Test: 15958 samples

  2. Downloaded and preprocessed Multilingual Toxic dataset
     - Train: 53099 samples
     - Test: 13275 samples

  3. Loaded pre-trained models from Hugging Face
     - English model      : pshashid/xlmr-toxicity-english/xlmr_toxicity_english.bin
     - Multilingual model : pshashid/xlmr-toxicity-multilingual/xlmr_toxicity_multilingual.bin

  4. Evaluated both models on test sets
     - English toxicity classification metrics
     - English intent classification metrics
     - Multilingual toxicity classification metrics
     - Per-language F1 scores

  5. Performed inference on sample texts
     - English toxic/clean examples
     - Multilingual examples (German, Spanish, French, Italian)
     - Batch inference on custom text list
PIPELINE COMPLETE!
